In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def prepare_data(features_df, binary=True, chronological=False, selected_features=None, test_size=0.2, scaler_type=None, random_state=42):

    if chronological:
        train_dfs = []
        test_dfs = []
    
        for class_id in sorted(features_df["Класс"].unique()):
            class_data = features_df[features_df["Класс"] == class_id]
            split_idx = int(len(class_data) * (1 - test_size))
    
            train_dfs.append(class_data.iloc[:split_idx])
            test_dfs.append(class_data.iloc[split_idx:])
    
        train_df = pd.concat(train_dfs).reset_index(drop=True)
        test_df = pd.concat(test_dfs).reset_index(drop=True)
    
        X_train = train_df.drop(columns=["Класс"])
        y_train = train_df["Класс"].values
    
        X_test = test_df.drop(columns=["Класс"])
        y_test = test_df["Класс"].values

    else:
        X = features_df.drop(columns=["Класс"])
        y = features_df["Класс"].values
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

    if selected_features is not None:
        selected_features = [f for f in selected_features if f in X_train.columns]
        X_train = X_train[selected_features]
        X_test = X_test[selected_features]

    if binary:
        y_train = (y_train != 0).astype(int)
        y_test = (y_test != 0).astype(int)

    if scaler_type == "standard":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test


def train_and_evaluate(models, results, key, X_train_raw, X_test_raw, X_train_std, X_test_std, y_train, y_test):

    for model_name, model in models.items():

        if model_name in ["K-Nearest Neighbors", "Logistic Regression"]:
            X_train, X_test = X_train_std, X_test_std
        else:
            X_train, X_test = X_train_raw, X_test_raw

        print("-" * 50)
        print(model_name)

        # Обучение
        start = time.perf_counter()
        model.fit(X_train, y_train)
        train_time = time.perf_counter() - start

        # Предсказание
        start = time.perf_counter()
        y_pred = model.predict(X_test)
        infer_time = time.perf_counter() - start

        # Метрики
        acc = accuracy_score(y_test, y_pred) * 100
        prec = precision_score(y_test, y_pred, average="weighted") * 100
        rec = recall_score(y_test, y_pred, average="weighted") * 100
        f1 = f1_score(y_test, y_pred, average="weighted") * 100

        time_per_pred = infer_time / len(y_test) * 1000

        results[key][model_name] = {
            "Accuracy": round(acc, 3),
            "Precision": round(prec, 3),
            "Recall": round(rec, 3),
            "F1_Score": round(f1, 3),
            "Training_Time": round(train_time, 3),
            "Inference_Time": round(infer_time, 4),
            "Time_per_pred_ms": round(time_per_pred, 4),
            "Confusion_Matrix": confusion_matrix(y_test, y_pred).tolist()
        }

        print(f"Accuracy: {acc:.2f}%")
        print(classification_report(y_test, y_pred))

    return results

### Пример использования

In [ ]:
overlaps = [50, 75]
window_sizes = [64, 32, 16, 8]

with open("DATA/rfecv_results/rfecv_threshold_0.98.json", 'r') as f:
    file = json.load(f)

features = file["optimal_features"][:50]

file_names = [name for name in os.listdir("DATA/features")
              if name.endswith('.csv') and
              any(f"ws{w}" in name for w in window_sizes) and
              any(f"overlap{o}" in name for o in overlaps)]

file_names = sorted(
    file_names,
    key=lambda x: (-int(x.split('_')[2].replace('ws', '')),
                   int(x.split('_')[3].replace('overlap', '').replace('.csv', '')))
)

results = {}

for file_name in file_names:

    print(f"\n{'='*50}")
    print(f"ФАЙЛ: {file_name}")
    print('='*50)

    df = pd.read_csv(f"DATA/features/{file_name}")

    # Без масштабирования
    X_train_raw, X_test_raw, y_train, y_test = prepare_data(
        df,
        selected_features=features,
        binary=True,
        scaler_type=None
    )

    # Со StandardScaler
    X_train_std, X_test_std, _, _ = prepare_data(
        df,
        selected_features=features,
        binary=True,
        scaler_type="standard"
    )

    models = {
        "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=37, p=1, metric='manhattan'),
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Histogram-based Gradient Boosting": HistGradientBoostingClassifier(),
        "XGBoost": XGBClassifier(device="cuda", tree_method="hist", n_jobs=-1),
        "LightGBM": LGBMClassifier(verbose=-1)
    }

    results[file_name] = {}

    results = train_and_evaluate(
        models,
        results,
        file_name,
        X_train_raw,
        X_test_raw,
        X_train_std,
        X_test_std,
        y_train,
        y_test
    )


with open('DATA/related_info/models_binary_top50_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4, cls=NumpyEncoder, ensure_ascii=False)

## Для мат. модели

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def prepare_data(features_df, binary=True, chronological=False, selected_features=None, test_size=0.2, scaler_type=None, random_state=42):
    

    
    if chronological:
        train_dfs = []
        test_dfs = []
    
        for class_id in sorted(features_df["Класс"].unique()):
            class_data = features_df[features_df["Класс"] == class_id]
            split_idx = int(len(class_data) * (1 - test_size))
    
            train_dfs.append(class_data.iloc[:split_idx])
            test_dfs.append(class_data.iloc[split_idx:])
    
        train_df = pd.concat(train_dfs).reset_index(drop=True)
        test_df = pd.concat(test_dfs).reset_index(drop=True)
    
        X_train = train_df.drop(columns=["Класс"])
        y_train = train_df["Класс"].values
    
        X_test = test_df.drop(columns=["Класс"])
        y_test = test_df["Класс"].values

    else:
        X = features_df.drop(columns=["Класс"])
        y = features_df["Класс"].values
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

    if binary:
        y_train = (y_train < 1).astype(int)
        y_test = (y_test < 1).astype(int)

    else:
        class_mapping = {val: i for i, val in enumerate(sorted(np.unique(y_train), reverse=True))}
        y_train = np.array([class_mapping[val] for val in y_train])
        y_test = np.array([class_mapping[val] for val in y_test])

    if selected_features is not None:
        selected_features = [f for f in selected_features if f in X_train.columns]
        X_train = X_train[selected_features]
        X_test = X_test[selected_features]

    if scaler_type == "standard":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test

